# Part 1 – Multi-Armed Bandit: Adaptive Treatment Recommendation System
### BITS Pilani WILP – Deep Reinforcement Learning | Lab Assignment 1
**Group Number: 32**

In [ ]:
import datetime, platform, socket, os

print("=" * 60)
print("  EXECUTION METADATA")
print("=" * 60)
print(f"Timestamp       : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Hostname        : {socket.gethostname()}")
print(f"Platform        : {platform.platform()}")
print(f"Python Version  : {platform.python_version()}")
# Virtual Machine ID – reads machine-id if available (Linux virtual labs)
try:
    with open("/etc/machine-id") as f:
        vm_id = f.read().strip()
except Exception:
    vm_id = "N/A (not a Linux VM or file missing)"
print(f"Virtual Machine ID: {vm_id}")
print(f"Working Directory : {os.getcwd()}")
print("=" * 60)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math, warnings
warnings.filterwarnings("ignore")

# Reproducibility seeds as per assignment rules
G = 32
import random
random.seed(G)
np.random.seed(G)

print(f"Group Number G = {G}")


## Task 1 – Dataset Design (1 Mark)
Generate the synthetic patient-treatment environment based on group number G = 32.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 1: Synthetic Environment Generation
# ──────────────────────────────────────────────────────────────────────────────

# Number of medicines (arms)
K = (G % 3) + 5
print(f"Group Number       : G = {G}")
print(f"Number of Medicines: K = (G mod 3) + 5 = ({G} mod 3) + 5 = {K}")

# Hidden success probabilities for each medicine
hidden_probs = {}
print("\nHidden Success Probabilities:")
for i in range(K):
    Pi = 0.4 + ((G + i) % 6) * 0.07
    hidden_probs[i] = round(Pi, 4)
    print(f"  Medicine {i}: P{i} = 0.4 + (({G}+{i}) mod 6) × 0.07 = {Pi:.4f}")

best_medicine = max(hidden_probs, key=hidden_probs.get)
print(f"\nOptimal medicine (ground truth): Medicine {best_medicine} with P = {hidden_probs[best_medicine]:.4f}")

# ──────────────────────────────────────────────────────────────────────────────
# Build base dataset with patient_id and severity_score only.
# assigned_medicine, clinical_outcome, utility_score are populated per algorithm.
# ──────────────────────────────────────────────────────────────────────────────
NUM_PATIENTS = 1000

base_df = pd.DataFrame({
    "patient_id"     : np.arange(NUM_PATIENTS),
    "severity_score" : [(pid % 5) + 1 for pid in range(NUM_PATIENTS)],
    "assigned_medicine": [np.nan] * NUM_PATIENTS,
    "clinical_outcome" : [np.nan] * NUM_PATIENTS,
    "utility_score"    : [np.nan] * NUM_PATIENTS,
})

print("\nFirst 10 rows of the base dataset:")
print(base_df.head(10).to_string(index=False))
print(f"\nTotal patients: {len(base_df)}")
print(f"Severity distribution: {base_df['severity_score'].value_counts().sort_index().to_dict()}")


## Helper Functions

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Helper: simulate a single patient interaction for a given medicine arm.
# Returns (clinical_outcome, utility_score).
# ──────────────────────────────────────────────────────────────────────────────

def treat_patient(patient_id, medicine_id, hidden_probs):
    """
    Simulate treating a patient with a given medicine.

    Parameters
    ----------
    patient_id   : int  – index of the patient (0-999)
    medicine_id  : int  – chosen medicine arm (0 to K-1)
    hidden_probs : dict – mapping from medicine_id to success probability

    Returns
    -------
    clinical_outcome : int   – 1 (recovered) or 0 (not recovered)
    utility_score    : float – reward accounting for disease severity
    """
    severity = (patient_id % 5) + 1                      # severity in {1,2,3,4,5}
    Pi = hidden_probs[medicine_id]                       # hidden success prob
    clinical_outcome = int(np.random.rand() < Pi)        # Bernoulli draw
    utility_score = clinical_outcome * (1 - severity / 10.0)
    return clinical_outcome, round(utility_score, 4)


def run_simulation(strategy_fn, hidden_probs, K, num_patients=1000):
    """
    Run a full 1000-patient simulation using the provided strategy function.

    Parameters
    ----------
    strategy_fn  : callable – bandit strategy that takes (t, counts, values) and
                              returns chosen arm index
    hidden_probs : dict     – hidden success probabilities
    K            : int      – number of medicine arms
    num_patients : int      – total iterations

    Returns
    -------
    df          : pd.DataFrame  – completed patient log
    cum_rewards : np.ndarray    – cumulative utility scores over patients
    """
    df = base_df.copy()
    counts = np.zeros(K)    # number of times each arm was selected
    values = np.zeros(K)    # estimated mean utility of each arm
    cum_rewards = np.zeros(num_patients)

    for t in range(num_patients):
        # Choose medicine using the provided strategy
        arm = strategy_fn(t, counts, values)

        # Simulate patient outcome
        clinical, utility = treat_patient(t, arm, hidden_probs)

        # Update arm statistics
        counts[arm] += 1
        values[arm] += (utility - values[arm]) / counts[arm]   # incremental mean

        # Log to dataframe
        df.at[t, "assigned_medicine"]  = arm
        df.at[t, "clinical_outcome"]   = clinical
        df.at[t, "utility_score"]      = utility

        # Track cumulative reward
        cum_rewards[t] = (cum_rewards[t-1] if t > 0 else 0) + utility

    return df, cum_rewards, counts, values


## Task 2 – Immediate Exploitation Strategy (1 Mark)
Test each medicine exactly 10 times initially (exploration phase), then always choose the best-performing medicine.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 2: Greedy Exploitation Strategy
# Policy: Explore each arm 10 times → exploit the best arm forever.
# ──────────────────────────────────────────────────────────────────────────────

# Reset seeds for reproducibility before each simulation
random.seed(G); np.random.seed(G)

INITIAL_TRIALS = 10   # number of initial trials per medicine

def greedy_strategy(t, counts, values):
    """
    Greedy (Immediate Exploitation) strategy.

    - Phase 1 (t < K * INITIAL_TRIALS): round-robin each arm exactly INITIAL_TRIALS times.
    - Phase 2 (t >= K * INITIAL_TRIALS): always pick the arm with the highest estimated value.

    Parameters
    ----------
    t      : int         – current iteration (0-indexed)
    counts : np.ndarray  – pull counts per arm
    values : np.ndarray  – estimated mean utility per arm

    Returns
    -------
    int : chosen arm index
    """
    exploration_end = K * INITIAL_TRIALS  # e.g., 7 * 10 = 70
    if t < exploration_end:
        # Round-robin: try arm t mod K during the exploration phase
        return t % K
    else:
        # Pure exploitation: choose the arm with the highest estimated mean utility
        return int(np.argmax(values))


df_greedy, cum_greedy, counts_greedy, values_greedy = run_simulation(
    greedy_strategy, hidden_probs, K
)

print("=" * 55)
print("  GREEDY (IMMEDIATE EXPLOITATION) RESULTS")
print("=" * 55)
print(f"Exploration phase : first {K * INITIAL_TRIALS} patients (10 per medicine)")
print(f"Exploitation phase: patients {K * INITIAL_TRIALS} – 999")
print()
print("Arm statistics after 1000 patients:")
for i in range(K):
    print(f"  Medicine {i}: pulled {int(counts_greedy[i]):4d} times | "
          f"est. utility = {values_greedy[i]:.4f} | true P = {hidden_probs[i]:.4f}")
print()
best_greedy = int(np.argmax(values_greedy))
print(f"Selected best medicine: {best_greedy} (true P = {hidden_probs[best_greedy]:.4f})")
print(f"Total cumulative reward : {cum_greedy[-1]:.4f}")
print()
print("Sample of patient log (first 15 rows):")
print(df_greedy[["patient_id","severity_score","assigned_medicine",
                  "clinical_outcome","utility_score"]].head(15).to_string(index=False))


## Task 3 – Controlled Clinical Trial Strategy / Epsilon-Greedy (1.5 Marks)
Explore with probability ε, otherwise exploit the best arm. Analyse ε = 0.10, 0.01, 0.50.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 3: Epsilon-Greedy Strategy
# ε% of patients receive a randomly chosen medicine (exploration),
# (1-ε)% receive the currently best medicine (exploitation).
# ──────────────────────────────────────────────────────────────────────────────

def make_epsilon_greedy(epsilon):
    """
    Factory function that returns an epsilon-greedy strategy closure.

    Parameters
    ----------
    epsilon : float – probability of choosing a random arm (exploration rate)

    Returns
    -------
    callable : strategy function compatible with run_simulation()
    """
    def strategy(t, counts, values):
        """
        Epsilon-Greedy bandit strategy.

        With probability epsilon  → pick a random arm (explore).
        With probability 1-epsilon → pick the arm with highest estimated value (exploit).
        """
        if np.random.rand() < epsilon:
            return np.random.randint(0, K)   # random exploration
        else:
            # If an arm has never been tried, choose it first to avoid zero-bias
            untried = np.where(counts == 0)[0]
            if len(untried) > 0:
                return untried[0]
            return int(np.argmax(values))    # exploit best known arm
    return strategy


epsilon_configs = {
    "ε=0.10 (10%)": 0.10,
    "ε=0.01 (1%)" : 0.01,
    "ε=0.50 (50%)": 0.50,
}

results_eps = {}
for label, eps in epsilon_configs.items():
    random.seed(G); np.random.seed(G)
    df_e, cum_e, cnt_e, val_e = run_simulation(make_epsilon_greedy(eps), hidden_probs, K)
    results_eps[label] = {"df": df_e, "cum": cum_e, "counts": cnt_e, "values": val_e}

print("=" * 65)
print("  EPSILON-GREEDY RESULTS COMPARISON")
print("=" * 65)
for label, res in results_eps.items():
    best_arm = int(np.argmax(res["values"]))
    print(f"\n{label}")
    print(f"  Final cumulative reward : {res['cum'][-1]:.4f}")
    print(f"  Best arm identified     : Medicine {best_arm} "
          f"(true P = {hidden_probs[best_arm]:.4f})")
    print(f"  Arm pull distribution   : "
          + ", ".join([f"M{i}:{int(res['counts'][i])}" for i in range(K)]))

print()
print("Detailed stats for ε=0.10 (primary strategy):")
res10 = results_eps["ε=0.10 (10%)"]
print(res10["df"][["patient_id","severity_score","assigned_medicine",
                    "clinical_outcome","utility_score"]].head(15).to_string(index=False))


## Task 4 – Confidence-Based Strategy / UCB1 (1 Mark)
Arms with fewer observations receive a higher exploration bonus that shrinks as evidence accumulates.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 4: UCB1 (Upper Confidence Bound) Strategy
#
# UCB1 score for arm i at time t:
#     UCB1(i) = estimated_value(i) + sqrt( 2 * ln(t) / counts(i) )
#
# Arms never pulled get +inf priority, ensuring each arm is tried once first.
# ──────────────────────────────────────────────────────────────────────────────

def ucb1_strategy(t, counts, values):
    """
    UCB1 (Upper Confidence Bound 1) strategy.

    Selects the arm that maximises: Q(a) + sqrt(2 * ln(t+1) / N(a))
    where Q(a) is the estimated mean reward and N(a) is the pull count.
    Unpulled arms are always prioritised (score = +infinity).

    Parameters
    ----------
    t      : int         – current iteration (0-indexed)
    counts : np.ndarray  – pull counts per arm
    values : np.ndarray  – estimated mean utility per arm

    Returns
    -------
    int : chosen arm index
    """
    # Always try unpulled arms first
    untried = np.where(counts == 0)[0]
    if len(untried) > 0:
        return untried[0]

    # Compute UCB1 score for each arm
    total_pulls = t + 1
    confidence_bonus = np.sqrt(2 * np.log(total_pulls) / counts)
    ucb_scores = values + confidence_bonus
    return int(np.argmax(ucb_scores))


random.seed(G); np.random.seed(G)
df_ucb, cum_ucb, counts_ucb, values_ucb = run_simulation(ucb1_strategy, hidden_probs, K)

print("=" * 55)
print("  UCB1 RESULTS")
print("=" * 55)
print("Arm statistics after 1000 patients:")
for i in range(K):
    bonus = np.sqrt(2 * np.log(1000) / max(counts_ucb[i], 1))
    print(f"  Medicine {i}: pulled {int(counts_ucb[i]):4d} | "
          f"est. utility = {values_ucb[i]:.4f} | "
          f"final UCB bonus = {bonus:.4f} | true P = {hidden_probs[i]:.4f}")

best_ucb = int(np.argmax(values_ucb))
print(f"\nBest arm identified : Medicine {best_ucb} (true P = {hidden_probs[best_ucb]:.4f})")
print(f"Total cumulative reward : {cum_ucb[-1]:.4f}")
print()
print("Sample of patient log (first 15 rows):")
print(df_ucb[["patient_id","severity_score","assigned_medicine",
              "clinical_outcome","utility_score"]].head(15).to_string(index=False))


## Task 5 – Comparative Analysis (0.5 Marks)
Plot cumulative reward vs number of patients for all strategies and answer the analysis questions.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 5: Comparative Analysis – Cumulative Reward vs Number of Patients
# ──────────────────────────────────────────────────────────────────────────────

patients = np.arange(1, 1001)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Plot 1: All strategies ────────────────────────────────────────────────────
ax1 = axes[0]
ax1.plot(patients, cum_greedy,
         label="Greedy (Exploitation)", color="steelblue", linewidth=2)
ax1.plot(patients, results_eps["ε=0.10 (10%)"]["cum"],
         label="ε-Greedy ε=0.10", color="darkorange", linewidth=2)
ax1.plot(patients, results_eps["ε=0.01 (1%)"]["cum"],
         label="ε-Greedy ε=0.01", color="green", linewidth=2)
ax1.plot(patients, results_eps["ε=0.50 (50%)"]["cum"],
         label="ε-Greedy ε=0.50", color="red", linewidth=2, linestyle="--")
ax1.plot(patients, cum_ucb,
         label="UCB1", color="purple", linewidth=2, linestyle="-.")
ax1.set_title("Cumulative Reward vs Number of Patients (All Strategies - Group 32)",
              fontsize=13, fontweight="bold")
ax1.set_xlabel("Number of Patients", fontsize=12)
ax1.set_ylabel("Cumulative Utility Score", fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# ── Plot 2: Epsilon comparison only ──────────────────────────────────────────
ax2 = axes[1]
colors = ["darkorange", "green", "red"]
for (label, res), color in zip(results_eps.items(), colors):
    ax2.plot(patients, res["cum"], label=label, color=color, linewidth=2)
ax2.set_title("Epsilon-Greedy: Effect of Exploration Rate (Group 32)", fontsize=13, fontweight="bold")
ax2.set_xlabel("Number of Patients", fontsize=12)
ax2.set_ylabel("Cumulative Utility Score", fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("MAB_comparative_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved as MAB_comparative_analysis.png")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 5: Summary Statistics Table
# ──────────────────────────────────────────────────────────────────────────────

strategies = {
    "Greedy (Exploitation)"  : cum_greedy,
    "ε-Greedy ε=0.10 (10%)" : results_eps["ε=0.10 (10%)"]["cum"],
    "ε-Greedy ε=0.01 (1%)"  : results_eps["ε=0.01 (1%)"]["cum"],
    "ε-Greedy ε=0.50 (50%)" : results_eps["ε=0.50 (50%)"]["cum"],
    "UCB1"                   : cum_ucb,
}

print(f"{'Strategy':<28} {'Final Reward':>14} {'Mean/Patient':>13} {'Std Dev':>10}")
print("-" * 68)
for name, cum in strategies.items():
    rewards_per_patient = np.diff(np.concatenate([[0], cum]))
    print(f"{name:<28} {cum[-1]:>14.4f} {np.mean(rewards_per_patient):>13.4f} "
          f"{np.std(rewards_per_patient):>10.4f}")


## Task 5 – Answers to Analysis Questions

### Q1. Which strategy achieves the highest cumulative reward at the end of 1000 patients?
**UCB1** typically achieves the highest cumulative reward because it balances exploration and exploitation in a mathematically principled way using confidence bounds, ensuring that the optimal arm (Medicine 3, P=0.75) is identified and exploited efficiently.

### Q2. Which strategy identifies the best medicine fastest (earliest convergence)?
**Greedy (Exploitation)** converges fastest — it locks onto the best-looking arm after just K×10 = 70 initial trials. However, this rapid lock-in can be premature if the initial sample was noisy. **UCB1** converges nearly as fast but more reliably.

### Q3. Which strategy shows the most stable performance over time (least fluctuations)?
**Greedy (Exploitation)** after convergence shows the most stable curve since it always prescribes the same medicine. Among adaptive strategies, **ε-Greedy ε=0.01** is also very stable because only 1% of decisions are random.

### Q4. Which strategy would you recommend for real-world hospital deployment? Justify.
**UCB1** is recommended for real-world deployment. It provides a theoretically guaranteed regret bound (O(log N)), adapts automatically without requiring a manually tuned ε, and naturally reduces exploration as evidence accumulates — making it both safe and effective. Unlike pure greedy approaches, UCB1 avoids premature commitment to a suboptimal treatment.

### Comparative Summary
The Greedy strategy converges quickly but risks locking onto a suboptimal treatment if the initial 70 trials are noisy. ε-Greedy with ε=0.10 strikes a reasonable balance, though the fixed exploration rate wastes resources by testing inferior medicines indefinitely. Very low ε (0.01) rarely explores and may miss a better medicine, while high ε (0.50) sacrifices too much reward to exploration. UCB1 outperforms all strategies in the long run by automatically adjusting exploration based on statistical confidence — making it the gold standard for clinical treatment recommendation under uncertainty.
